# 🎙️ Satang TTS Studio — Google Colab Control Center (F5-TTS Commercial)
Production-grade Thai Custom TTS training targeting ElevenLabs parity on Colab Free T4 (15GB VRAM) with F5-TTS Backbone (MIT Commercial 100%), Multi-Speaker Management, Social Ingestion (YT/FB/IG), Target Voice Filter, and Interactive Studio Playground.

## [STEP 1] ⚙️ System Setup & Base Model Manager (F5-TTS 100% Commercial)
Mount Google Drive, install dependencies via `uv` in < 60s, and ensure F5-TTS base model weights are permanently cached in Drive.

In [ ]:
#@title 1.1 Mount Drive & Install Environment { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["LOG_LEVEL"] = "DEBUG"
os.environ["LOG_FILE"] = "/content/drive/MyDrive/tts-project/logs/pipeline_debug.log"

# Install uv package manager (takes ~1s)
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

# Clone repo or navigate into project directory
!git clone https://github.com/SatangThevalue/satangthevalue-tts-custom.git /content/satangthevalue-tts-custom 2>/dev/null || true
%cd /content/satangthevalue-tts-custom
!git pull origin main

# Install dependencies with uv (or fallback to standard pip)
!/root/.local/bin/uv pip install --system -e . || pip install -e .
!/root/.local/bin/uv pip install --system pythainlp faster-whisper pedalboard yt-dlp scipy soundfile phonemizer pypinyin || pip install pythainlp faster-whisper pedalboard yt-dlp scipy soundfile phonemizer pypinyin

# Auto-create required Google Drive directory structure
!mkdir -p /content/drive/MyDrive/tts-project/{00_base_models,01_raw,02_processed,03_checkpoints,04_onnx_exports,logs}

# Cache Base Model in Google Drive
!python -m src.models.downloader --model f5-tts --base-dir "/content/drive/MyDrive/tts-project/00_base_models"

# Run Pipeline Sanity Self-Check
!python tests/test_master_pipeline.py


In [ ]:
#@title 1.2 🔄 Quick Drive Sync & Cache Refresh (กดรันได้ทุกเมื่อ) { display-mode: "form" }
active_speaker = "satang" #@param {type:"string"}

import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("src.") or mod == "src":
        del sys.modules[mod]

from src.utils.drive_sync import print_drive_status_dashboard
print_drive_status_dashboard(
    base_dir="/content/drive/MyDrive/tts-project",
    speaker_id=active_speaker
)


## [STEP 2] 📥 Ingestion & Speaker Manager
เลือกใช้งานได้ตามสะดวก:
* **ใช้เซลล์ 2.1** หากต้องการดาวน์โหลดเสียงจาก YouTube / Facebook / Instagram
* **ใช้เซลล์ 2.2** หากต้องการอัปโหลดไฟล์เสียง (.wav, .mp3, .m4a) ตรงจากเครื่องคอมพิวเตอร์หรือมือถือ

In [ ]:
#@title 2.1 🌐 Download from Social Media (YouTube / Facebook / Instagram) { display-mode: "form" }
speaker_name = "satang" #@param {type:"string"}
social_urls = "https://www.youtube.com/watch?v=EXAMPLE" #@param {type:"string"}
max_parallel_workers = 3 #@param {type:"slider", min:1, max:6, step:1}
refresh_drive = True #@param {type:"boolean"}

# Invalidate cached Python modules to guarantee latest GitHub code is loaded
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("src.") or mod == "src":
        del sys.modules[mod]

from pathlib import Path
from src.audio.download import download_media_audio
from src.utils.drive_sync import refresh_drive_directory

speaker_raw_dir = Path(f"/content/drive/MyDrive/tts-project/01_raw/{speaker_name}")
speaker_raw_dir.mkdir(parents=True, exist_ok=True)
if refresh_drive:
    refresh_drive_directory(str(speaker_raw_dir))

# Support multiple URLs (separated by newlines or commas)
raw_items = social_urls.replace(",", "\n").split("\n")
urls = [u.strip() for u in raw_items if u.strip() and "EXAMPLE" not in u and not u.strip().startswith("#")]

if urls:
    print(f"🌐 กำลังดาวน์โหลด {len(urls)} รายการแบบ Parallel (Workers: {max_parallel_workers}) สำหรับ '{speaker_name}'...")
    res = download_media_audio(
        urls,
        output_dir="/content/drive/MyDrive/tts-project/01_raw",
        speaker_id=speaker_name,
        max_workers=max_parallel_workers
    )
    success = sum(1 for r in res if r.get('status') in ('success', 'cached'))
    print(f"✅ ดาวน์โหลดสำเร็จ {success}/{len(urls)} รายการ!")
else:
    print("⚠️ กรุณาระบุลิงก์ในช่อง social_urls (ใส่หลายลิงก์ได้ โดยคั่นด้วยเครื่องหมายจุลภาค หรือขึ้นบรรทัดใหม่)")

current_files = refresh_drive_directory(str(speaker_raw_dir))
print(f"\n📁 ไฟล์เสียงดิบใน {speaker_raw_dir} ({len(current_files)} ไฟล์):")
for f_str in sorted(current_files)[:10]:
    f = Path(f_str)
    print(f"   • {f.name} ({f.stat().st_size / (1024*1024):.2f} MB)")


In [ ]:
#@title 2.2 📁 Upload Audio Files from Device (PC / Mobile) { display-mode: "form" }
speaker_name = "satang" #@param {type:"string"}
refresh_drive = True #@param {type:"boolean"}

from google.colab import files
from pathlib import Path
from src.utils.drive_sync import refresh_drive_directory

speaker_raw_dir = Path(f"/content/drive/MyDrive/tts-project/01_raw/{speaker_name}")
speaker_raw_dir.mkdir(parents=True, exist_ok=True)
if refresh_drive:
    refresh_drive_directory(str(speaker_raw_dir))

print(f"📥 กำลังเปิดหน้าต่างเลือกไฟล์สำหรับ Speaker: '{speaker_name}'...")
uploaded = files.upload()
if uploaded:
    for fname, data in uploaded.items():
        dest = speaker_raw_dir / fname
        with open(dest, "wb") as f:
            f.write(data)
        print(f"✅ บันทึกไฟล์สำเร็จ -> {dest.name} ({len(data)/(1024*1024):.2f} MB)")
else:
    print("⚠️ ไม่ได้เลือกไฟล์")

current_files = refresh_drive_directory(str(speaker_raw_dir))
print(f"\n📁 ไฟล์เสียงดิบใน {speaker_raw_dir} ({len(current_files)} ไฟล์):")
for f_str in sorted(current_files)[:10]:
    f = Path(f_str)
    print(f"   • {f.name} ({f.stat().st_size / (1024*1024):.2f} MB)")


## [STEP 3] 🧹 Audio Restoration (Enhance 24kHz) & VAD Slicing (3-10s)
Restores mobile audio quality using Resemble Enhance (Denoise/De-reverb), slices into optimal 3–10s chunks with breath preservation, and compares Before vs After.

In [ ]:
#@title 3.1 Run Enhance & Silero Slicer { display-mode: "form" }
target_speaker = "satang" #@param {type:"string"}
refresh_drive = True #@param {type:"boolean"}

from pathlib import Path
from src.utils.drive_sync import refresh_drive_directory

raw_dir = Path(f"/content/drive/MyDrive/tts-project/01_raw/{target_speaker}")
if refresh_drive:
    refresh_drive_directory(str(raw_dir))

raw_files = list(raw_dir.glob("*.*"))
if not raw_files:
    print(f"⚠️ ไม่พบไฟล์เสียงดิบใน {raw_dir}")
    print("💡 กรุณาดาวน์โหลดคลิป (Cell 2.1) หรืออัปโหลดไฟล์เสียง (Cell 2.2) เข้ามาก่อนเริ่มขั้นตอนนี้ครับ")
else:
    print(f"🎙️ พบไฟล์เสียงดิบ {len(raw_files)} ไฟล์ กำลังเริ่มกู้สภาพเสียง (Enhance 24kHz)...")
    # 3.1.1 Enhance phone recordings to 24kHz studio grade
    !python -m src.audio.enhance \
        --raw-dir "/content/drive/MyDrive/tts-project/01_raw/{target_speaker}" \
        --out-dir "/content/drive/MyDrive/tts-project/02_processed/enhanced/{target_speaker}"

    # 3.1.2 Silero VAD Slicing with breath preservation (3-10s)
    !python -m src.audio.slicer \
        --input "/content/drive/MyDrive/tts-project/02_processed/enhanced/{target_speaker}" \
        --out-dir "/content/drive/MyDrive/tts-project/02_processed/wavs/{target_speaker}"

    # Audio Player Comparison (Preview first 15s to prevent browser buffer truncation on long podcasts)
    from IPython.display import Audio, display
    import soundfile as sf

    raw_samples = sorted(list(raw_dir.glob("*.wav")))
    enhanced_samples = sorted(list(Path(f"/content/drive/MyDrive/tts-project/02_processed/enhanced/{target_speaker}").glob("*.wav")))
    sliced_chunks = sorted(list(Path(f"/content/drive/MyDrive/tts-project/02_processed/wavs/{target_speaker}").glob("*.wav")))

    if raw_samples and enhanced_samples:
        try:
            # Read only first 15 seconds to ensure fast, safe in-browser playback
            r_data, r_sr = sf.read(str(raw_samples[0]), stop=24000 * 15)
            e_data, e_sr = sf.read(str(enhanced_samples[0]), stop=24000 * 15)
            print(f"\n🔊 ตัวอย่างเสียงดิบ 15 วินาทีแรก ({raw_samples[0].name}):")
            display(Audio(r_data, rate=r_sr))
            print(f"✨ ตัวอย่างเสียงหลัง Enhance 15 วินาทีแรก ({enhanced_samples[0].name}):")
            display(Audio(e_data, rate=e_sr))
        except Exception as e:
            print(f"⚠️ Audio Player warning: {e}")

    if sliced_chunks:
        print(f"\n✂️ ตัวอย่างท่อนเสียงที่หั่นเสร็จแล้วพร้อมนำไปเทรน ({sliced_chunks[0].name}):")
        display(Audio(str(sliced_chunks[0])))


## [STEP 4] 📝 Faster-Whisper ASR & Thai Tonal G2P Reviewer
Auto-transcribes sliced speech with Faster-Whisper Large-v3 and applies Thai Tonal locking [0-4] with phonemizer.

In [ ]:
#@title 4.1 Transcribe & Generate Tonal Metadata { display-mode: "form" }
from pathlib import Path

# Auto-heal: verify required packages for Whisper ASR & Thai G2P
try:
    import faster_whisper
    import pythainlp
    import av
except ImportError:
    print("📦 กำลังติดตั้งแพ็กเกจ faster-whisper, pythainlp และ av ให้อัตโนมัติ...")
    !pip install -q pythainlp faster-whisper phonemizer pypinyin "av>=12.0.0"

wavs_dir = Path("/content/drive/MyDrive/tts-project/02_processed/wavs")
found_wavs = list(wavs_dir.rglob("*.wav"))
if not found_wavs:
    print(f"⚠️ ไม่พบไฟล์ท่อนเสียงใน {wavs_dir}")
    print("💡 กรุณารัน Step 3.1 (Enhance & Silero Slicer) เพื่อหั่นไฟล์เสียงก่อนครับ")
else:
    print(f"🎙️ พบท่อนเสียงทั้งหมด {len(found_wavs)} ท่อน กำลังเริ่มถอดเสียงภาษาไทยและล็อกวรรณยุกต์ [0-4]...")
    # 4.1 Auto ASR (Permanent Google Drive model caching at 00_base_models/whisper)
    !python -m src.asr.transcribe \
        --wavs-dir "/content/drive/MyDrive/tts-project/02_processed/wavs" \
        --output-jsonl "/content/drive/MyDrive/tts-project/02_processed/metadata_raw.jsonl" \
        --download-root "/content/drive/MyDrive/tts-project/00_base_models/whisper"

    # 4.2 Thai Tone Tagging [0-4]
    !python -m src.g2p.text_norm \
        --input "/content/drive/MyDrive/tts-project/02_processed/metadata_raw.jsonl" \
        --output "/content/drive/MyDrive/tts-project/02_processed/metadata.jsonl"

    # Transcript Previewer
    import json
    from IPython.display import Audio, display
    print("\n🔍 ตัวอย่างผลการถอดเสียงและล็อกวรรณยุกต์:")
    meta_p = Path("/content/drive/MyDrive/tts-project/02_processed/metadata.jsonl")
    if meta_p.exists():
        with open(meta_p, 'r', encoding='utf-8') as f:
            for idx, line in enumerate(f):
                if idx >= 3: break
                rec = json.loads(line)
                print(f"[{idx+1}] ข้อความ: {rec.get('normalized_text')}")
                print(f"    หน่วยเสียง (Tone Lock): {rec.get('phonemes')}")
                if Path(rec.get("audio_path", "")).exists():
                    display(Audio(rec["audio_path"]))


## [STEP 5] 📊 Speaker Audit Dashboard (Pre-flight Gate)
Inspect total speech duration, word frequencies, lexical diversity (TTR), and Thai tone coverage [0-4] before starting fine-tuning.

In [ ]:
#@title 5.1 Run Pre-flight Audit Dashboard { display-mode: "form" }
audit_speaker = "satang" #@param {type:"string"}
min_minutes_required = 30.0 #@param {type:"number"}
refresh_drive = True #@param {type:"boolean"}

from src.utils.drive_sync import refresh_drive_directory
if refresh_drive:
    refresh_drive_directory("/content/drive/MyDrive/tts-project/02_processed")

!python -m src.utils.speaker_inspector \
    --metadata "/content/drive/MyDrive/tts-project/02_processed/metadata.jsonl" \
    --speaker "{audit_speaker}" \
    --min-minutes {min_minutes_required}


## [STEP 6] 🚀 LoRA Fine-Tuning Control Center (F5-TTS Backbone)
Memory-safe 8-bit training (< 11GB VRAM) with automatic checkpoint sync to Google Drive every 200 steps and auto-resume.

In [ ]:
#@title 6.1 Train LoRA Voice Adapter { display-mode: "form" }
train_speaker = "satang" #@param {type:"string"}
training_steps = 2500 #@param {type:"integer"}
batch_size = 2 #@param {type:"integer"}

from pathlib import Path
meta_check = Path("/content/drive/MyDrive/tts-project/02_processed/metadata.jsonl")
if not meta_check.exists():
    print(f"⚠️ ไม่พบไฟล์ {meta_check}")
    print("💡 กรุณารัน Step 4.1 (ถอดเสียงและสร้าง metadata) ให้เสร็จสมบูรณ์ก่อนเริ่มเทรนครับ")
else:
    # Pack dataset to Colab local NVMe for lightning-fast training I/O
    from src.utils.pack import pack_dataset, unpack_dataset
    tar_path = "/content/drive/MyDrive/tts-project/02_processed/dataset.tar"
    pack_dataset("/content/drive/MyDrive/tts-project/02_processed", tar_path)
    unpack_dataset(tar_path, "/content/dataset_local")

    # Start LoRA training with isolated speaker checkpointing
    !python -m src.training.finetune_lora \
        --config configs/base_config.yaml \
        --lora-config configs/lora_config.yaml \
        --speaker-id "{train_speaker}"


In [ ]:
#@title 6.2 🎧 Pre-Export Sound Check (ทดสอบฟังเสียงจาก Checkpoint ก่อนแปลง ONNX) { display-mode: "form" }
test_speaker = "satang" #@param {type:"string"}
eval_step = 0 #@param {type:"integer"}
test_prompt = "สวัสดีครับ นี่คือการทดสอบเสียงจากโมเดลโดยตรง ก่อนตัดสินใจแปลงเป็นไฟล์ ONNX" #@param {type:"string"}
speed = 1.0 #@param {type:"slider", min:0.7, max:1.4, step:0.05}
pitch = 0.0 #@param {type:"slider", min:-3.0, max:3.0, step:0.5}
refresh_drive = True #@param {type:"boolean"}

from IPython.display import Audio, display
from src.inference.test_checkpoint import preview_checkpoint
from src.utils.drive_sync import refresh_drive_directory

if refresh_drive:
    refresh_drive_directory(f"/content/drive/MyDrive/tts-project/03_checkpoints/{test_speaker}")

try:
    print(f"🎙️ กำลังทดสอบเสียงจาก Checkpoint ของ '{test_speaker}' (Step: {'ล่าสุด' if eval_step == 0 else eval_step})...")
    res = preview_checkpoint(
        speaker_id=test_speaker,
        text=test_prompt,
        checkpoint_step=eval_step,
        output_wav="checkpoint_preview.wav",
        speed_factor=speed,
        pitch_semitones=pitch,
    )
    print(f"🎉 สร้างเสียงตัวอย่างสำเร็จจาก Step {res['checkpoint_step']} ใน {res['elapsed_sec']:.2f} วินาที!")
    print(f"Phonemes: {res['phonemes']}")
    display(Audio(res['output_path']))
except Exception as e:
    print(f"⚠️ แจ้งเตือน: {e}")


## [STEP 7] Model Version Control and Studio Voice Tuning
* **Cell 7.1:** Register model version tag (e.g. v1.0, v1.1) and export to ONNX INT8 with database tracking
* **Cell 7.2:** Studio Voice Tuner with Interactive Sliders (Speed, Pitch, Warmth, Reverb, De-esser)

In [ ]:
#@title 7.1 Register Model Version and Export ONNX INT8 { display-mode: "form" }
export_speaker = "satang" #@param {type:"string"}
version_tag = "v1.0" #@param {type:"string"}
checkpoint_step = 2500 #@param {type:"integer"}
version_notes = "Production release 24kHz studio" #@param {type:"string"}

import os
from pathlib import Path
from src.utils.registry import register_model_version, list_model_versions

ckpt_file = Path(f"/content/drive/MyDrive/tts-project/03_checkpoints/{export_speaker}/step_{checkpoint_step}/adapter_model.pt")
if not ckpt_file.exists():
    avail_ckpts = sorted(list(Path(f"/content/drive/MyDrive/tts-project/03_checkpoints/{export_speaker}").glob("step_*")))
    print(f"⚠️ ไม่พบไฟล์เช็คพอยต์: {ckpt_file}")
    if avail_ckpts:
        print("💡 เช็คพอยต์ที่มีอยู่ให้เลือกใช้งาน:")
        for c in avail_ckpts:
            print(f"   • {c.name}")
    else:
        print("💡 ยังไม่มีเช็คพอยต์ใดๆ ในโฟลเดอร์ กรุณารัน Step 6.1 (เทรนโมเดล) ก่อนครับ")
else:
    export_dir = Path(f"/content/drive/MyDrive/tts-project/04_onnx_exports/{export_speaker}/{version_tag}")
    export_dir.mkdir(parents=True, exist_ok=True)
    raw_onnx = str(export_dir / "model.onnx")
    quant_onnx = str(export_dir / "model_quant_int8.onnx")

    print(f"📦 กำลังแปลงโมเดล Step {checkpoint_step} -> {export_dir}...")
    !python -m src.export.export_onnx \
        --checkpoint "{str(ckpt_file)}" \
        --output "{raw_onnx}" \
        --speaker-id "{export_speaker}"

    !python -m src.export.optimize \
        --input "{raw_onnx}" \
        --output "{quant_onnx}"

    register_model_version(
        speaker_id=export_speaker,
        version_tag=version_tag,
        checkpoint_step=checkpoint_step,
        onnx_path=raw_onnx,
        quant_onnx_path=quant_onnx,
        notes=version_notes
    )

    print(f"\n📜 รายการเวอร์ชันโมเดลที่บันทึกไว้สำหรับ '{export_speaker}':")
    versions = list_model_versions(speaker_id=export_speaker)
    for v in versions:
        print(f"   • [{v['version_tag']}] Step: {v['checkpoint_step']} | File: {Path(v['quant_onnx_path']).name} | Note: {v['notes']}")


In [ ]:
#@title 7.2 Studio Voice Playground with Custom Tuner Sliders { display-mode: "form" }
speaker_name = "satang" #@param {type:"string"}
model_version = "v1.0" #@param {type:"string"}
test_text = "สวัสดีครับ นี่คือเสียงสังเคราะห์ระบบภาษาไทย คุณภาพสูงระดับสตูดิโอ ปรับแต่งเสียงได้ตามใจชอบ" #@param {type:"string"}

# Voice Tuning Controls
speed_factor = 1.0 #@param {type:"slider", min:0.7, max:1.5, step:0.05}
pitch_semitones = 0.0 #@param {type:"slider", min:-4.0, max:4.0, step:0.5}
warmth_drive = 1.15 #@param {type:"slider", min:1.0, max:1.5, step:0.05}
room_reverb = 0.04 #@param {type:"slider", min:0.0, max:0.2, step:0.01}
deess_reduction = -2.5 #@param {type:"slider", min:-6.0, max:0.0, step:0.5}

from IPython.display import Audio, display
from pathlib import Path
from src.inference.engine import TTSEngine

onnx_path = f"/content/drive/MyDrive/tts-project/04_onnx_exports/{speaker_name}/{model_version}/model_quant_int8.onnx"
fallback_onnx = f"/content/drive/MyDrive/tts-project/04_onnx_exports/{speaker_name}/model_quant_int8.onnx"

active_model = onnx_path if Path(onnx_path).exists() else fallback_onnx

if Path(active_model).exists():
    print(f"Generating with model [{model_version}]...")
    print(f"Settings: Speed={speed_factor}x | Pitch={pitch_semitones:+} st | Warmth={warmth_drive} | Reverb={room_reverb}")
    engine = TTSEngine(active_model)
    result = engine.synthesize(
        text=test_text,
        output_wav_path="studio_custom_tuned.wav",
        enable_mastering=True,
        speed_factor=speed_factor,
        pitch_semitones=pitch_semitones,
        warmth_drive=warmth_drive,
        room_reverb_wet=room_reverb,
        deess_gain_db=deess_reduction,
    )
    print(f"Done in {result['elapsed_sec']:.2f}s (RTF: {result['rtf']:.3f})")
    print(f"Phonemes: {result['phonemes']}")
    display(Audio(result['output_path']))
else:
    print(f"Model not found at {active_model}. Please run cell 7.1 to export first.")
